# Leaflet cluster map of talk locations

Run this from the _talks/ directory, which contains .md files of all your talks. This scrapes the location YAML field from each .md file, geolocates it with geopy/Nominatim, and uses the getorg library to output data, HTML, and Javascript for a standalone cluster map.

In [54]:
!pip install getorg --upgrade ipywidgets
import glob
import re
import shutil
from pathlib import Path

import getorg
from geopy import Nominatim
from geopy.extra.rate_limiter import RateLimiter
from geopy.extra.rate_limiter import RateLimiter


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: C:\Users\siser\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [55]:
repo_root = Path.cwd()
for candidate in [repo_root, *repo_root.parents]:
    if (candidate / "_talks").exists() and (candidate / "talkmap").exists():
        repo_root = candidate
        break

talk_dir = repo_root / "_talks"
output_dir = repo_root / "talkmap"
other_locations_file = repo_root / "scripts" / "other-locations.txt"

g = sorted(talk_dir.glob("*.md"))

In [56]:
geocoder = Nominatim(user_agent="siserte-talkmap", timeout=20)
rate_limited_geocode = RateLimiter(
    geocoder.geocode,
    min_delay_seconds=1.0,
    max_retries=5,
    error_wait_seconds=5.0,
    swallow_exceptions=False,
)
location_dict = {}
location = ""
permalink = ""
title = ""
location_status = []

location_overrides = {
    "Grenoble (France)": "Grenoble, France",
    "PPAM26, Poznan (Poland)": "Poznan, Poland",
    "Poznan (Poland)": "Poznan, Poland",
    "Politechnika Częstochowska, Poland": "Częstochowa, Poland",
    "Technical University of Munich, Garching near Munich, Germany": "Munich, Germany",
    "ISC25, Hamburg, Germany": "Hamburg, Germany",
    "Garching (Germany)": "Garching, Germany",
    "Barcelona, Spain": "Barcelona, Spain",
    "Madrid, Spain": "Madrid, Spain",
    "Paris, France": "Paris, France",
    "Rome, Italy": "Rome, Italy",
    "Bristol, United Kingdom": "Bristol, United Kingdom",
    "Málaga, Spain": "Málaga, Spain",
    "Lugano, Switzerland": "Lugano, Switzerland",
    "Rennes, France": "Rennes, France",
    "Denver, Colorado, USA": "Denver, Colorado, USA",
    "Ostrava, Czech Republic": "Ostrava, Czech Republic",
    "Kobe, Japan": "Kobe, Japan",
    "Bordeaux, France": "Bordeaux, France",
    "Ciudad Real, Spain": "Ciudad Real, Spain",
    "Limassol, Cyprus": "Limassol, Cyprus",
    "Castelló de la Plana, Spain": "Castelló de la Plana, Spain",
    "Córdoba, Spain": "Córdoba, Spain",
    "Timisoara, Romania": "Timișoara, Romania",
    "Hamburg, Germany": "Hamburg, Germany",
    "Lugano, Switzerland": "Lugano, Switzerland",
    "Valladolid, Spain": "Valladolid, Spain",
    "Teruel, Spain": "Teruel, Spain",
    "Częstochowa, Poland": "Częstochowa, Poland",
}


def record_status(raw_value, normalized_value, status, reason=""):
    location_status.append({
        "raw": raw_value,
        "normalized": normalized_value,
        "status": status,
        "reason": reason,
    })


def normalize_location(value):
    if not value:
        return ""
    cleaned = value.strip().replace("’", "'")
    if cleaned in location_overrides:
        return location_overrides[cleaned]
    cleaned = re.sub(r"\s*\(([^)]+)\)\s*$", r", \1", cleaned)
    cleaned = re.sub(r"^[A-Z0-9]+,\s*", "", cleaned)
    cleaned = re.sub(r"\s+near\s+.*$", "", cleaned)
    return cleaned.strip()


def geocode_location(value):
    if not value or value == "Online":
        return None

    normalized = normalize_location(value)
    if not normalized:
        return None

    try:
        result = rate_limited_geocode(normalized)
        if result is None:
            print(f"->SKIPPED: no geocode for {value!r} (normalized: {normalized!r})")
            return None
        return result
    except Exception as exc:
        print(f"->SKIPPED: geocoding failed for {value!r} (normalized: {normalized!r}): {exc}")
        return None

In [57]:
for file in g:
    with open(file, 'r', encoding="utf-8") as f:
        lines = f.read()
        if 'location: "' in lines:
            loc_start = lines.find('location: "') + 11
            lines_trim = lines[loc_start:]
            loc_end = lines_trim.find('"')
            location = lines_trim[:loc_end]
            if "Teruel" in location:
                print("->IGNORED:", location, "\n")
                record_status(location, normalize_location(location), "pass", "ignored")
            elif location and location not in location_dict and location != "Online":
                normalized = normalize_location(location)
                if normalized:
                    geocode = geocode_location(normalized)
                    if geocode is not None:
                        location_dict[normalized] = geocode
                        record_status(location, normalized, "pass", "geocoded")
                        print(normalized, "\n", geocode)
                    else:
                        record_status(location, normalized, "fail", "geocode_failed")
                else:
                    record_status(location, "", "fail", "empty_normalized_value")
            else:
                record_status(location, normalize_location(location), "pass", "online_or_duplicate")

RateLimiter caught an error, retrying (0/5 tries). Called with (*('Madrid, Spain',), **{}).
Traceback (most recent call last):
  File "C:\Users\siser\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\geopy\geocoders\base.py", line 368, in _call_geocoder
    result = self.adapter.get_json(url, timeout=timeout, headers=req_headers)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\siser\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\geopy\adapters.py", line 472, in get_json
    resp = self._request(url, timeout=timeout, headers=headers)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\siser\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\geopy\adapters.py", line 500, in _request
    raise 

->SKIPPED: geocoding failed for 'Madrid, Spain' (normalized: 'Madrid, Spain'): Non-successful status code 429


RateLimiter caught an error, retrying (0/5 tries). Called with (*('Valladolid, Spain',), **{}).
Traceback (most recent call last):
  File "C:\Users\siser\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\geopy\geocoders\base.py", line 368, in _call_geocoder
    result = self.adapter.get_json(url, timeout=timeout, headers=req_headers)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\siser\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\geopy\adapters.py", line 472, in get_json
    resp = self._request(url, timeout=timeout, headers=headers)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\siser\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\geopy\adapters.py", line 500, in _request
    ra

Valladolid, Spain 
 Valladolid, Castilla y León, España
Lugano, Switzerland 
 Lugano, Circolo di Lugano ovest, Distretto di Lugano, Ticino, Schweiz/Suisse/Svizzera/Svizra
Paris, France 
 Paris, Île-de-France, France métropolitaine, France
Barcelona, Spain 
 Barcelona, Barcelonès, Barcelona, Catalunya, España
Castelló de la Plana, Spain 
 Castelló de la Plana / Castellón de la Plana, la Plana Alta, Castelló / Castellón, Comunitat Valenciana, España
Córdoba, Spain 
 Córdoba, Andalucía, España
Timișoara, Romania 
 Timișoara, Timiș, România
Rome, Italy 
 Roma, Roma Capitale, Lazio, Italia
Bristol, United Kingdom 
 City of Bristol, West of England, England, United Kingdom
Málaga, Spain 
 Málaga, Málaga-Costa del Sol, Málaga, Andalucía, España
->IGNORED: Teruel, Spain 

Limassol, Cyprus 
 Λεμεσός, Δήμος Λεμεσού, Επαρχία Λεμεσού, Κύπρος, 3085, Κύπρος - Kıbrıs
Rennes, France 
 Rennes, Ille-et-Vilaine, Bretagne, France métropolitaine, France
Częstochowa, Poland 
 Częstochowa, województwo śląski

In [58]:
if other_locations_file.exists():
    with open(other_locations_file, 'r', encoding="utf-8") as f:
        while line := f.readline():
            location = line.rstrip()
            if location and location not in location_dict and location != "Online":
                normalized = normalize_location(location)
                if normalized:
                    geocode = geocode_location(normalized)
                    if geocode is not None:
                        location_dict[normalized] = geocode
                        record_status(location, normalized, "pass", "geocoded")
                        print(normalized, "\n", geocode)
                    else:
                        record_status(location, normalized, "fail", "geocode_failed")
                else:
                    record_status(location, "", "fail", "empty_normalized_value")
            else:
                record_status(location, normalize_location(location), "pass", "online_or_duplicate")

Belfast, Northern Ireland, UK 
 Belfast City District, County Antrim, Northern Ireland / Tuaisceart Éireann, United Kingdom
Berkeley, California, USA 
 Berkeley, Alameda County, California, United States
St. Charles, Illinois, USA 
 St. Charles, St. Charles Township, Kane County, Illinois, 60174, United States
Fiuggi, Italy 
 Fiuggi, Frosinone, Lazio, 03014, Italia
Amsterdam, Netherlands 
 Amsterdam, Noord-Holland, Nederland
Ljubljana, Slovenia 
 Ljubljana, Upravna Enota Ljubljana, 1000, Slovenija
Dallas, Texas, USA 
 Dallas, Dallas County, Texas, United States
Salt Lake City, Utah, USA 
 Salt Lake City, Salt Lake County, Utah, United States
Austin, Texas, USA 
 Austin, Travis County, Texas, United States
Turin, Italy 
 Torino, Piemonte, Italia
Grenoble, France 
 Grenoble, Isère, Auvergne-Rhône-Alpes, France métropolitaine, France
València, Spain 
 València, Comarca de València, València / Valencia, Comunitat Valenciana, España
Almería, Spain 
 Almería, Andalucía, España
Garching, Germ

In [59]:
if output_dir.exists():
    shutil.rmtree(output_dir)
else:
    output_dir.mkdir(parents=True, exist_ok=True)

m = getorg.orgmap.create_map_obj()
getorg.orgmap.output_html_cluster_map(location_dict, folder_name=str(output_dir), hashed_usernames=False)
print(f"Created {len(location_dict)} geocoded locations in {output_dir}")

Created 51 geocoded locations in \\wsl.localhost\Ubuntu\home\siserte\siserte.github.io\talkmap


In [60]:
print(f"Unique geocoded locations: {len(location_dict)}")
print(f"Output folder: {output_dir}")
print(f"Pass: {sum(1 for item in location_status if item['status'] == 'pass')}")
print(f"Fail: {sum(1 for item in location_status if item['status'] == 'fail')}")
print("Skipped entries are printed above as ->SKIPPED or ->IGNORED")
print("Full status list: ")
for entry in location_status:
    print(entry)

Unique geocoded locations: 51
Output folder: \\wsl.localhost\Ubuntu\home\siserte\siserte.github.io\talkmap
Pass: 66
Fail: 1
Skipped entries are printed above as ->SKIPPED or ->IGNORED
Full status list: 
{'raw': 'Madrid, Spain', 'normalized': 'Madrid, Spain', 'status': 'fail', 'reason': 'geocode_failed'}
{'raw': 'Valladolid, Spain', 'normalized': 'Valladolid, Spain', 'status': 'pass', 'reason': 'geocoded'}
{'raw': 'Lugano, Switzerland', 'normalized': 'Lugano, Switzerland', 'status': 'pass', 'reason': 'geocoded'}
{'raw': 'Paris, France', 'normalized': 'Paris, France', 'status': 'pass', 'reason': 'geocoded'}
{'raw': 'Barcelona, Spain', 'normalized': 'Barcelona, Spain', 'status': 'pass', 'reason': 'geocoded'}
{'raw': 'Castelló de la Plana, Spain', 'normalized': 'Castelló de la Plana, Spain', 'status': 'pass', 'reason': 'geocoded'}
{'raw': 'Córdoba, Spain', 'normalized': 'Córdoba, Spain', 'status': 'pass', 'reason': 'geocoded'}
{'raw': 'Timisoara, Romania', 'normalized': 'Timișoara, Romania